# Overture POI Type Clusters and MTUS Activity Mask Draft

This notebook prepares a first-pass POI-type mask for micro-activity assignment.

Goal for this pass:

- collect unique Overture Maps place `categories.primary` values;
- cluster the POI types into a smaller set of semantic buckets;
- propose which MTUS micro-activities are acceptable for each bucket;
- write reviewable CSVs without changing alignment training or scoring.

The default source is the checked-in `citybehavex/category/unique_categories.csv`, which is fast and stable. Set `USE_OVERTURE_S3 = True` below to scan an Overture release directly and attach global POI counts.

In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 80)

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "citybehavex").exists():
            return candidate
    raise RuntimeError("Could not find repository root")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from citybehavex.activities.catalog import build_catalog

OUTPUT_DIR = REPO_ROOT / "notebooks" / "05_poi_activity_mask" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_CSV = REPO_ROOT / "citybehavex" / "category" / "unique_categories.csv"
LOCAL_TESSELLATION_PATHS = [
    REPO_ROOT / "data" / "gparis" / "results" / "gparis_poi_tessellation.parquet",
    REPO_ROOT / "data" / "shanghai" / "results" / "shanghai_tessellation.parquet",
    REPO_ROOT / "data" / "yjmob" / "results" / "yjmob_tessellation.parquet",
    REPO_ROOT / "data" / "yjmob2" / "results" / "yjmob2_tessellation.parquet",
]

OVERTURE_RELEASE = "2026-05-20.0"
USE_OVERTURE_S3 = False
N_CLUSTERS = 18

REPO_ROOT, OUTPUT_DIR

(PosixPath('/home/gustavo/citybehavex'),
 PosixPath('/home/gustavo/citybehavex/notebooks/05_poi_activity_mask/outputs'))

## Load Unique POI Types

The local CSV contains the known Overture primary categories and the current broad-purpose labels used by tessellation. If `USE_OVERTURE_S3` is enabled, the notebook scans the Overture Places parquet release and replaces/augments these rows with observed global counts.

In [2]:
def normalize_label(value: object) -> str:
    return " ".join(str(value).strip().lower().replace("&", " and ").replace("_", " ").split())

def load_local_categories(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["primary_category"] = df["primary_category"].astype(str).str.strip()
    df["purpose"] = df["purpose"].fillna("OTHER").astype(str).str.upper()
    df = df.drop_duplicates("primary_category").sort_values("primary_category").reset_index(drop=True)
    df["category_text"] = df["primary_category"].map(normalize_label)
    df["source"] = "citybehavex/category/unique_categories.csv"
    return df

def fetch_overture_primary_categories(release: str) -> pd.DataFrame:
    import duckdb

    query = f"""
        INSTALL httpfs; LOAD httpfs;
        SET s3_region = 'us-west-2';

        SELECT
            categories.primary AS primary_category,
            COUNT(*) AS overture_poi_count
        FROM read_parquet(
            's3://overturemaps-us-west-2/release/{release}/theme=places/type=place/*',
            filename=true,
            hive_partitioning=1
        )
        WHERE categories.primary IS NOT NULL
        GROUP BY 1
        ORDER BY overture_poi_count DESC, primary_category
    """
    df = duckdb.sql(query).df()
    df["primary_category"] = df["primary_category"].astype(str).str.strip()
    df["category_text"] = df["primary_category"].map(normalize_label)
    df["source"] = f"overture:{release}"
    return df

categories = load_local_categories(CATEGORY_CSV)

if USE_OVERTURE_S3:
    overture_categories = fetch_overture_primary_categories(OVERTURE_RELEASE)
    categories = overture_categories.merge(
        categories[["primary_category", "purpose"]],
        on="primary_category",
        how="left",
    )
    categories["purpose"] = categories["purpose"].fillna("OTHER")

display(categories.head(20))
print(f"Unique primary categories: {len(categories):,}")
print(categories["purpose"].value_counts(dropna=False).to_string())

,primary_category,purpose,category_text,source
0,abuse_and_addiction_treatment,HEALTH,abuse and addiction treatment,citybehavex/category/unique_categories.csv
1,academic_bookstore,PURCHASE,academic bookstore,citybehavex/category/unique_categories.csv
2,accommodation,OTHER,accommodation,citybehavex/category/unique_categories.csv
3,accountant,OTHER,accountant,citybehavex/category/unique_categories.csv
4,active_life,LEISURE,active life,citybehavex/category/unique_categories.csv
5,acupuncture,HEALTH,acupuncture,citybehavex/category/unique_categories.csv
6,adult_education,STUDIES,adult education,citybehavex/category/unique_categories.csv
7,adult_entertainment,LEISURE,adult entertainment,citybehavex/category/unique_categories.csv
8,adult_store,PURCHASE,adult store,citybehavex/category/unique_categories.csv
9,advertising_agency,WORK,advertising agency,citybehavex/category/unique_categories.csv


Unique primary categories: 1,114
purpose
LEISURE     366
PURCHASE    305
OTHER       191
WORK        118
HEALTH       90
STUDIES      44


## Optional Local Observed Counts

If a generated POI tessellation exists locally, attach observed category counts from it. These counts are not required for the semantic clustering, but they help prioritize review.

In [3]:
def read_local_category_counts(paths: list[Path]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in paths:
        if not path.exists():
            continue
        df = pd.read_parquet(path, columns=["category"])
        if "category" not in df.columns:
            continue
        counts = (
            df["category"]
            .dropna()
            .astype(str)
            .value_counts()
            .rename_axis("primary_category")
            .reset_index(name=path.stem)
        )
        frames.append(counts)
    if not frames:
        return pd.DataFrame(columns=["primary_category"])
    out = frames[0]
    for frame in frames[1:]:
        out = out.merge(frame, on="primary_category", how="outer")
    count_cols = [c for c in out.columns if c != "primary_category"]
    out[count_cols] = out[count_cols].fillna(0).astype(int)
    out["local_observed_count"] = out[count_cols].sum(axis=1)
    return out

local_counts = read_local_category_counts(LOCAL_TESSELLATION_PATHS)
categories = categories.merge(local_counts, on="primary_category", how="left")
count_cols = [c for c in categories.columns if c.endswith("_tessellation") or c == "local_observed_count"]
if count_cols:
    categories[count_cols] = categories[count_cols].fillna(0).astype(int)

sort_col = "overture_poi_count" if "overture_poi_count" in categories.columns else "local_observed_count"
if sort_col in categories.columns:
    display(categories.sort_values(sort_col, ascending=False).head(30))
else:
    display(categories.head(30))

,primary_category,purpose,category_text,source,gparis_poi_tessellation,local_observed_count
823,professional_services,PURCHASE,professional services,citybehavex/category/unique_categories.csv,8557,8557
437,french_restaurant,LEISURE,french restaurant,citybehavex/category/unique_categories.csv,7197,7197
256,community_services_non_profits,OTHER,community services non profits,citybehavex/category/unique_categories.csv,7163,7163
869,restaurant,LEISURE,restaurant,citybehavex/category/unique_categories.csv,6790,6790
536,hotel,LEISURE,hotel,citybehavex/category/unique_categories.csv,5972,5972
127,beauty_salon,PURCHASE,beauty salon,citybehavex/category/unique_categories.csv,5506,5506
239,clothing_store,PURCHASE,clothing store,citybehavex/category/unique_categories.csv,4859,4859
483,hair_salon,PURCHASE,hair salon,citybehavex/category/unique_categories.csv,4528,4528
109,banks,OTHER,banks,citybehavex/category/unique_categories.csv,4377,4377
976,supermarket,PURCHASE,supermarket,citybehavex/category/unique_categories.csv,4347,4347


## Semantic Clustering

This uses lightweight TF-IDF over category names. It is intentionally transparent: the cluster labels below are heuristic and meant to be reviewed, not treated as a learned model contract.

In [4]:
vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    token_pattern=r"(?u)\b[a-z][a-z0-9]+\b",
    min_df=1,
    sublinear_tf=True,
)
X = vectorizer.fit_transform(categories["category_text"])

clusterer = AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="cosine", linkage="average")
categories["cluster_id"] = clusterer.fit_predict(X.toarray())

terms = np.array(vectorizer.get_feature_names_out())
centroids = []
for cluster_id in sorted(categories["cluster_id"].unique()):
    rows = np.where(categories["cluster_id"].to_numpy() == cluster_id)[0]
    mean_vector = np.asarray(X[rows].mean(axis=0)).ravel()
    top_terms = terms[np.argsort(mean_vector)[::-1][:8]].tolist()
    centroids.append({"cluster_id": cluster_id, "top_terms": ", ".join(top_terms), "n_categories": len(rows)})

cluster_terms = pd.DataFrame(centroids).sort_values(["n_categories", "cluster_id"], ascending=[False, True])
display(cluster_terms)

,cluster_id,top_terms,n_categories
1,1,"restaurant, store, and, services, service, shop, school, bar",883
0,0,"church, airport, temple, gym, lounge, pub, boxing, therapist",210
2,2,"installation, countertop installation, countertop, solar installation, solar, windows, windows installation, electric",3
3,3,"designer, web, web designer, graphic designer, graphic, architectural, architectural designer, electric",3
5,5,"zoo, petting zoo, petting, electric utility, electric, egyptian restaurant, egyptian, educational supply store",2
4,4,"winery, electric utility, electric, egyptian restaurant, egyptian, educational supply store, educational supply, edu...",1
6,6,"well, well drilling, drilling, electrical, electric utility provider, electric utility, electric, egyptian restaurant",1
7,7,"nan, electric utility provider, electric utility, electric, egyptian restaurant, egyptian, educational supply store,...",1
8,8,"waxing, electric utility provider, electric utility, electric, egyptian restaurant, egyptian, educational supply sto...",1
9,9,"waterfall, electric utility provider, electric utility, electric, egyptian restaurant, egyptian, educational supply ...",1


In [5]:
CLUSTER_NAME_RULES = [
    ("food_drink", r"restaurant|cafe|coffee|tea|bar|pub|bakery|bistro|diner|pizza|burger|food|drink|ice cream|dessert|brewery|winery"),
    ("retail_shopping", r"store|shop|market|mall|retail|bookstore|boutique|supermarket|grocery|pharmacy|florist|jewelry|clothing|electronics|furniture"),
    ("personal_services", r"salon|spa|barber|beauty|massage|repair|laundry|dry cleaner|tailor|service|rental|bank|atm|accountant|lawyer|insurance"),
    ("health_care", r"hospital|clinic|doctor|dentist|medical|health|therapy|veterinar|nursing|pharmacy|optometrist|chiropractor|acupuncture|treatment"),
    ("education", r"school|university|college|education|class|academy|training|tutor|library|study|kindergarten|daycare"),
    ("work_industry", r"manufacturer|factory|industrial|company|agency|office|contractor|supplier|warehouse|logistics|construction|agriculture|farm|plant"),
    ("religion_civic", r"church|mosque|temple|synagogue|religious|worship|charity|nonprofit|civic|government|embassy|city hall|courthouse"),
    ("sports_fitness", r"sport|fitness|gym|yoga|pilates|martial|swim|tennis|golf|stadium|arena|recreation|climbing|dance|skate"),
    ("culture_events", r"museum|gallery|theater|cinema|movie|music|concert|auditorium|arts|entertainment|nightclub|event|casino|arcade"),
    ("parks_outdoors", r"park|garden|trail|beach|camp|zoo|aquarium|nature|outdoor|attraction|amusement"),
    ("transport_travel", r"airport|station|terminal|transit|parking|taxi|bus|rail|train|airline|hotel|lodging|accommodation|travel"),
    ("auto_mobility", r"auto|car|vehicle|motorcycle|gas station|fuel|charging|mechanic|body shop|dealer|tire|garage"),
    ("home_residential", r"apartment|residential|real estate|home|housing|assisted living|retirement|property"),
    ("care_family", r"child|children|daycare|elder|senior|family|pet|animal|veterinar|shelter"),
]

def semantic_bucket(text: str, fallback_cluster_id: int) -> str:
    for name, pattern in CLUSTER_NAME_RULES:
        if re.search(pattern, text):
            return name
    return "other_mixed"

categories["semantic_cluster"] = [
    semantic_bucket(text, cluster_id)
    for text, cluster_id in zip(categories["category_text"], categories["cluster_id"])
]

cluster_summary = (
    categories
    .groupby("semantic_cluster")
    .agg(
        n_categories=("primary_category", "nunique"),
        example_categories=("primary_category", lambda s: ", ".join(s.dropna().astype(str).head(12))),
    )
    .sort_values("n_categories", ascending=False)
    .reset_index()
)

display(cluster_summary)

,semantic_cluster,n_categories,example_categories
0,other_mixed,337,"active_life, agricultural_cooperatives, alternative_medicine, anesthesiologist, animation_studio, archery_range, arc..."
1,food_drink,188,"afghan_restaurant, african_restaurant, amateur_sports_team, american_restaurant, arabian_restaurant, argentine_resta..."
2,personal_services,140,"abuse_and_addiction_treatment, accountant, agricultural_service, aircraft_repair, alcohol_and_drug_treatment_centers..."
3,retail_shopping,121,"academic_bookstore, adult_store, antique_store, appliance_store, aquatic_pet_store, archery_shop, audio_visual_equip..."
4,work_industry,54,"advertising_agency, agriculture, aircraft_manufacturer, appliance_manufacturer, auto_company, auto_manufacturers_and..."
5,sports_fitness,45,"amateur_sports_league, atv_recreation_park, baseball_stadium, basketball_stadium, boxing_gym, country_dance_hall, da..."
6,culture_events,35,"adult_entertainment, arcade, art_gallery, art_museum, arts_and_crafts, arts_and_entertainment, asian_art_museum, aud..."
7,education,33,"adult_education, art_school, boxing_class, circus_school, college_university, cooking_school, cosmetology_school, cy..."
8,transport_travel,31,"accommodation, airline, airlines, airport, airport_lounge, airport_shuttles, airport_terminal, bus_station, bus_tour..."
9,health_care,31,"acupuncture, aromatherapy, cannabis_clinic, childrens_hospital, chiropractor, clinical_laboratories, cosmetic_dentis..."


## MTUS Activity Catalog

`missing` is kept visible for completeness but excluded from proposed masks by default.

In [6]:
activities = pd.DataFrame(
    {
        "activity_idx": a.idx,
        "activity": a.name,
        "description": a.description,
        "eligible_purposes": ",".join(map(str, a.eligible_purposes)),
    }
    for a in build_catalog()
)
display(activities)

,activity_idx,activity,description,eligible_purposes
0,0,sleep,Sleeping or resting at home,0
1,1,eatdrink,"Eating, drinking, coffee, lunch, and meal breaks","0,1,2"
2,2,selfcare,"Personal hygiene, grooming, and private care","0,2"
3,3,paidwork,Working at the office or job site,1
4,4,educatn,"Studying, attending class, or doing homework",2
5,5,foodprep,Cooking and food preparation,0
6,6,cleanetc,"Cleaning, laundry, and other domestic work",0
7,7,maintain,"Household maintenance, repairs, and administrative upkeep",0
8,8,shopserv,Shopping and personal services,2
9,9,garden,Gardening and outdoor household work,0


## First-Pass Cluster-to-Activity Rules

These rules are deliberately conservative. They are meant to reduce impossible choices at a POI without replacing the contextual alignment model. Edit `CLUSTER_ACTIVITY_RULES` and re-run the cells to revise the draft.

In [7]:
BASE_OTHER = {"eatdrink", "shopserv", "goout", "leisure"}

CLUSTER_ACTIVITY_RULES: dict[str, set[str]] = {
    "food_drink": {"eatdrink", "goout", "leisure", "shopserv", "read", "compint"},
    "retail_shopping": {"shopserv", "eatdrink", "leisure"},
    "personal_services": {"shopserv", "selfcare", "maintain"},
    "health_care": {"shopserv", "eldcare", "pkidcare", "selfcare"},
    "education": {"educatn", "read", "compint", "eatdrink"},
    "work_industry": {"paidwork", "eatdrink", "selfcare", "leisure", "compint"},
    "religion_civic": {"religion", "volorgwk", "leisure"},
    "sports_fitness": {"sportex", "selfcare", "leisure", "goout", "eatdrink"},
    "culture_events": {"goout", "leisure", "read", "eatdrink"},
    "parks_outdoors": {"sportex", "garden", "leisure", "pkidcare", "ikidcare", "petcare", "eatdrink"},
    "transport_travel": {"eatdrink", "shopserv", "read", "compint"},
    "auto_mobility": {"shopserv", "maintain", "read", "compint"},
    "home_residential": {"sleep", "eatdrink", "selfcare", "paidwork", "foodprep", "cleanetc", "maintain", "garden", "petcare", "eldcare", "pkidcare", "ikidcare", "tvradio", "read", "compint", "leisure"},
    "care_family": {"eldcare", "pkidcare", "ikidcare", "petcare", "shopserv", "leisure"},
}

def allowed_activities_for_cluster(cluster: str) -> set[str]:
    if cluster == "other_mixed":
        return set(BASE_OTHER)
    return set(CLUSTER_ACTIVITY_RULES.get(cluster, BASE_OTHER))

activity_names = activities[activities["activity"] != "missing"]["activity"].tolist()

cluster_activity_mask = pd.DataFrame(
    [
        {
            "semantic_cluster": cluster,
            "n_categories": int((categories["semantic_cluster"] == cluster).sum()),
            **{activity: activity in allowed_activities_for_cluster(cluster) for activity in activity_names},
        }
        for cluster in sorted(categories["semantic_cluster"].unique())
    ]
)

cluster_activity_long = cluster_activity_mask.melt(
    id_vars=["semantic_cluster", "n_categories"],
    var_name="activity",
    value_name="allowed",
)

display(cluster_activity_mask)

,semantic_cluster,n_categories,sleep,eatdrink,selfcare,paidwork,educatn,foodprep,cleanetc,maintain,...,religion,volorgwk,commute,travel,sportex,tvradio,read,compint,goout,leisure
0,auto_mobility,28,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,True,True,False,False
1,care_family,10,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,culture_events,35,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,True,True
3,education,33,False,True,False,False,True,False,False,False,...,False,False,False,False,False,False,True,True,False,False
4,food_drink,188,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,True,True,True
5,health_care,31,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
6,home_residential,15,True,True,True,True,False,True,True,True,...,False,False,False,False,False,True,True,True,False,True
7,other_mixed,338,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
8,parks_outdoors,27,False,True,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,True
9,personal_services,140,False,False,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


## Expand to Category-Level Mask

This is the candidate artifact for later simulation masking. It stays out of the training/scoring path for now.

In [8]:
category_activity_mask = categories[["primary_category", "category_text", "purpose", "cluster_id", "semantic_cluster"]].merge(
    cluster_activity_long,
    on="semantic_cluster",
    how="left",
)

wide_cols = ["primary_category", "category_text", "purpose", "cluster_id", "semantic_cluster"]
category_activity_wide = categories[wide_cols].copy()
for activity in activity_names:
    category_activity_wide[activity] = category_activity_wide["semantic_cluster"].map(
        lambda cluster: activity in allowed_activities_for_cluster(cluster)
    )

category_activity_wide["n_allowed_activities"] = category_activity_wide[activity_names].sum(axis=1)

display(category_activity_wide.sort_values(["semantic_cluster", "primary_category"]).head(80))
print(category_activity_wide["n_allowed_activities"].describe().to_string())

,primary_category,category_text,purpose,cluster_id,semantic_cluster,sleep,eatdrink,selfcare,paidwork,educatn,...,volorgwk,commute,travel,sportex,tvradio,read,compint,goout,leisure,n_allowed_activities
76,auto_customization,auto customization,OTHER,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
77,auto_detailing,auto detailing,PURCHASE,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
79,auto_loan_provider,auto loan provider,OTHER,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
82,auto_upholstery,auto upholstery,PURCHASE,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
84,automobile_leasing,automobile leasing,PURCHASE,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
86,automotive,automotive,PURCHASE,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
87,automotive_consultant,automotive consultant,WORK,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
88,automotive_dealer,automotive dealer,PURCHASE,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
92,automotive_storage_facility,automotive storage facility,WORK,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4
142,boat_dealer,boat dealer,PURCHASE,1,auto_mobility,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,4


count    1114.000000
mean        4.428187
std         1.739751
min         3.000000
25%         3.250000
50%         4.000000
75%         5.000000
max        16.000000


## Review Views

Use these quick tables to find clusters that are too broad, too narrow, or semantically weird.

In [9]:
review = (
    category_activity_wide
    .groupby("semantic_cluster")
    .agg(
        n_categories=("primary_category", "count"),
        min_allowed=("n_allowed_activities", "min"),
        max_allowed=("n_allowed_activities", "max"),
        examples=("primary_category", lambda s: ", ".join(s.dropna().astype(str).head(20))),
    )
    .sort_values("n_categories", ascending=False)
)
display(review)

activity_coverage = (
    category_activity_mask[category_activity_mask["allowed"]]
    .groupby("activity")
    .agg(n_categories=("primary_category", "nunique"))
    .sort_values("n_categories", ascending=False)
)
display(activity_coverage)

,n_categories,min_allowed,max_allowed,examples
semantic_cluster,,,,
other_mixed,337,4,4,"active_life, agricultural_cooperatives, alternative_medicine, anesthesiologist, animation_studio, archery_range, arc..."
food_drink,188,6,6,"afghan_restaurant, african_restaurant, amateur_sports_team, american_restaurant, arabian_restaurant, argentine_resta..."
personal_services,140,3,3,"abuse_and_addiction_treatment, accountant, agricultural_service, aircraft_repair, alcohol_and_drug_treatment_centers..."
retail_shopping,121,3,3,"academic_bookstore, adult_store, antique_store, appliance_store, aquatic_pet_store, archery_shop, audio_visual_equip..."
work_industry,54,5,5,"advertising_agency, agriculture, aircraft_manufacturer, appliance_manufacturer, auto_company, auto_manufacturers_and..."
sports_fitness,45,5,5,"amateur_sports_league, atv_recreation_park, baseball_stadium, basketball_stadium, boxing_gym, country_dance_hall, da..."
culture_events,35,4,4,"adult_entertainment, arcade, art_gallery, art_museum, arts_and_crafts, arts_and_entertainment, asian_art_museum, aud..."
education,33,4,4,"adult_education, art_school, boxing_class, circus_school, college_university, cooking_school, cosmetology_school, cy..."
transport_travel,31,4,4,"accommodation, airline, airlines, airport, airport_lounge, airport_shuttles, airport_terminal, bus_station, bus_tour..."


,n_categories
activity,
eatdrink,886
shopserv,886
leisure,850
goout,605
compint,349
read,330
selfcare,285
maintain,183
pkidcare,83


## Write Draft Artifacts

In [10]:
categories_out = OUTPUT_DIR / "overture_poi_type_clusters.csv"
cluster_mask_out = OUTPUT_DIR / "semantic_cluster_activity_mask.csv"
category_mask_out = OUTPUT_DIR / "overture_poi_type_activity_mask.csv"
category_mask_long_out = OUTPUT_DIR / "overture_poi_type_activity_mask_long.csv"

categories.sort_values(["semantic_cluster", "primary_category"]).to_csv(categories_out, index=False)
cluster_activity_mask.sort_values("semantic_cluster").to_csv(cluster_mask_out, index=False)
category_activity_wide.sort_values(["semantic_cluster", "primary_category"]).to_csv(category_mask_out, index=False)
category_activity_mask.sort_values(["semantic_cluster", "primary_category", "activity"]).to_csv(category_mask_long_out, index=False)

print(f"Wrote {categories_out.relative_to(REPO_ROOT)}")
print(f"Wrote {cluster_mask_out.relative_to(REPO_ROOT)}")
print(f"Wrote {category_mask_out.relative_to(REPO_ROOT)}")
print(f"Wrote {category_mask_long_out.relative_to(REPO_ROOT)}")

Wrote notebooks/05_poi_activity_mask/outputs/overture_poi_type_clusters.csv
Wrote notebooks/05_poi_activity_mask/outputs/semantic_cluster_activity_mask.csv
Wrote notebooks/05_poi_activity_mask/outputs/overture_poi_type_activity_mask.csv
Wrote notebooks/05_poi_activity_mask/outputs/overture_poi_type_activity_mask_long.csv
